In [1]:
import demes
import scipy
import dpluspy
import pandas 
import numpy as np
import moments
from collections import defaultdict

In [2]:
model_labels = {
    0: "null",
    1: "S->ND",
    2: "S->D",
    3: "H->N",
    4: "D<->N",
    5: "S->ND,S->D",
    6: "S->ND,H->N",
    7: "S->ND,D<->N",
    8: "S->D,H->N",
    9: "S->D,D<->N",
    10: "H->N,D<->N",
    11: "S->ND,S->D,H->N",
    12: "S->ND,S->D,D<->N",
    13: "S->ND,H->N,D<->N",
    14: "S->D,H->N,D<->N",
    15: "S->ND,S->D,H->N,D<->N"
}

table_builder = defaultdict(list)

for model in range(16):
    param_file = "models/ghost_model_0_yor1_params.yaml"
    lls = []
    for rep in range(30):
        file = f"fitted_models/ghost_model_{model}_Bherer_yor1_rep_{rep}.yaml"
        ll = demes.load(file).metadata["opt_info"]["ll"]
        lls.append(ll)
    table_builder["model"].append(model)
    table_builder["label"].append(model_labels[model])
    table_builder["ll"].append(np.max(lls))
    table_builder["best_fit"].append(np.argmax(lls))

tbl = pandas.DataFrame(table_builder)
tbl.to_csv("model_tbl.csv", index=False, sep="\t")

In [3]:
print(tbl.to_string(index=False))

 model                 label          ll  best_fit
     0                  null -627.186398         9
     1                 S->ND -622.269270         2
     2                  S->D -433.020382        17
     3                  H->N -399.900032        12
     4                 D<->N -604.906812         2
     5            S->ND,S->D -433.035515         1
     6            S->ND,H->N -393.709952        24
     7           S->ND,D<->N -602.693604        23
     8             S->D,H->N -389.124614         1
     9            S->D,D<->N -408.901920         9
    10            H->N,D<->N -394.134244         4
    11       S->ND,S->D,H->N -386.677133         6
    12      S->ND,S->D,D<->N -408.991943         3
    13      S->ND,H->N,D<->N -392.461686        10
    14       S->D,H->N,D<->N -386.950477        27
    15 S->ND,S->D,H->N,D<->N -385.428760        27


In [4]:
"""
Below we work through all nesting relationships in this set of models and 
evaluate a naive LRT score for each. In this cell are the naive LRT thresholds
for alpha = 0.05 and the relevant null distributions.
"""

x = 2.7056
sig = moments.Godambe.sum_chi2_ppf(x, (0.5, 0.5))
print(f"sf at {x} of 1/2 * chi2_0 + 1/2 * ch2_1: {sig:.6}")

x = 5.1384
sig = moments.Godambe.sum_chi2_ppf(x, (0, 0.5, 0.5))
print(f"sf at {x} of 1/2 * chi2_1 + 1/2 * ch2_2: {sig:.6}")

sf at 2.7056 of 1/2 * chi2_0 + 1/2 * ch2_1: 0.0499982
sf at 5.1384 of 1/2 * chi2_1 + 1/2 * ch2_2: 0.0499995


In [5]:
def run_lrt(_model1, _model0, df=None):
    """
    _model0 should be the index of the nested model.
    """
    ll0 = next(iter(tbl[tbl["model"] == _model0]["ll"]))
    ll1 = next(iter(tbl[tbl["model"] == _model1]["ll"]))
    lrt_stat = 2 * (ll1 - ll0)
    # sig = moments.Godambe.sum_chi2_ppf(lrt_stat, null_distr)
    sig = scipy.stats.chi2.sf(lrt_stat, df)
    label0 = next(iter(tbl[tbl["model"] == _model0]["label"]))
    label1 = next(iter(tbl[tbl["model"] == _model1]["label"]))
    print(f"LRT of model {_model1} [{label1}] against model {_model0} [{label0}]")
    print(f"D = {lrt_stat:.5}")
    print(f"p = {sig:.5}")

In [6]:
"""
Each feature finds support in isolation; Ghost introgression into ND is
relatively weakly supported. Support for the three other models is strong.
"""
run_lrt(1, 0, df=2)
run_lrt(2, 0, df=2)
run_lrt(3, 0, df=2)
run_lrt(4, 0, df=1)

LRT of model 1 [S->ND] against model 0 [null]
D = 9.8343
p = 0.0073201
LRT of model 2 [S->D] against model 0 [null]
D = 388.33
p = 4.729e-85
LRT of model 3 [H->N] against model 0 [null]
D = 454.57
p = 1.9534e-99
LRT of model 4 [D<->N] against model 0 [null]
D = 44.559
p = 2.4678e-11


In [ ]:
"""
Incorporating the feature S->ND on various submodel backgrounds. Confronting
a variety of submodels, this feature finds poor support, with significance only
when the null model just has the H->N pulse.
"""
run_lrt(5, 2, df=2)
run_lrt(6, 3, df=2)
run_lrt(7, 4, df=2)
run_lrt(11, 8, df=2)
run_lrt(13, 10, df=2)
run_lrt(12, 9, df=2)
run_lrt(15, 14, df=2)

LRT of model 5 [S->ND,S->D] against model 2 [S->D]
D = -0.030266
p = 1.0
LRT of model 6 [S->ND,H->N] against model 3 [H->N]
D = 12.38
p = 0.0020497
LRT of model 7 [S->ND,D<->N] against model 4 [D<->N]
D = 4.4264
p = 0.10935
LRT of model 11 [S->ND,S->D,H->N] against model 8 [S->D,H->N]
D = 4.895
p = 0.086511
LRT of model 13 [S->ND,H->N,D<->N] against model 10 [H->N,D<->N]
D = 3.3451
p = 0.18777
LRT of model 12 [S->ND,S->D,D<->N] against model 9 [S->D,D<->N]
D = -0.18005
p = 1.0
LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 14 [S->D,H->N,D<->N]
D = 3.0434
p = 0.21834


In [8]:
"""
Incorporating S->D. This feature finds fairly strong support across various
null models.
"""
run_lrt(5, 1, df=2)
run_lrt(8, 3, df=2)
run_lrt(9, 4, df=2)
run_lrt(14, 10, df=2)
run_lrt(11, 6, df=2)
run_lrt(12, 7, df=2)
run_lrt(15, 13, df=2)

LRT of model 5 [S->ND,S->D] against model 1 [S->ND]
D = 378.47
p = 6.5588e-83
LRT of model 8 [S->D,H->N] against model 3 [H->N]
D = 21.551
p = 2.0907e-05
LRT of model 9 [S->D,D<->N] against model 4 [D<->N]
D = 392.01
p = 7.5189e-86
LRT of model 14 [S->D,H->N,D<->N] against model 10 [H->N,D<->N]
D = 14.368
p = 0.0007588
LRT of model 11 [S->ND,S->D,H->N] against model 6 [S->ND,H->N]
D = 14.066
p = 0.00088244
LRT of model 12 [S->ND,S->D,D<->N] against model 7 [S->ND,D<->N]
D = 387.4
p = 7.5238e-85
LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 13 [S->ND,H->N,D<->N]
D = 14.066
p = 0.00088235


In [9]:
""" 
Incorporating H->N. This feature is overwhelmingly supported in these tests.
"""
run_lrt(6, 1, df=2)
run_lrt(8, 2, df=2)
run_lrt(10, 4, df=2)
run_lrt(14, 9, df=2)
run_lrt(11, 5, df=2) 
run_lrt(13, 7, df=2)
run_lrt(15, 12, df=2)

LRT of model 6 [S->ND,H->N] against model 1 [S->ND]
D = 457.12
p = 5.4695e-100
LRT of model 8 [S->D,H->N] against model 2 [S->D]
D = 87.792
p = 8.6359e-20
LRT of model 10 [H->N,D<->N] against model 4 [D<->N]
D = 421.55
p = 2.9016e-92
LRT of model 14 [S->D,H->N,D<->N] against model 9 [S->D,D<->N]
D = 43.903
p = 2.9283e-10
LRT of model 11 [S->ND,S->D,H->N] against model 5 [S->ND,S->D]
D = 92.717
p = 7.3589e-21
LRT of model 13 [S->ND,H->N,D<->N] against model 7 [S->ND,D<->N]
D = 420.46
p = 4.9824e-92
LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 12 [S->ND,S->D,D<->N]
D = 47.126
p = 5.8431e-11


In [10]:
""" 
Incorporating D<->N. This feature finds strong support in some cases, but has
borderline significance when S->D is modelled and is not significant with S->ND.
"""
run_lrt(7, 1, df=1)
run_lrt(9, 2, df=1)
run_lrt(10, 3, df=1)
run_lrt(12, 5, df=1) 
run_lrt(13, 6, df=1) 
run_lrt(14, 8, df=1) 
run_lrt(15, 11, df=1) 

LRT of model 7 [S->ND,D<->N] against model 1 [S->ND]
D = 39.151
p = 3.922e-10
LRT of model 9 [S->D,D<->N] against model 2 [S->D]
D = 48.237
p = 3.7771e-12
LRT of model 10 [H->N,D<->N] against model 3 [H->N]
D = 11.532
p = 0.00068424
LRT of model 12 [S->ND,S->D,D<->N] against model 5 [S->ND,S->D]
D = 48.087
p = 4.0769e-12
LRT of model 13 [S->ND,H->N,D<->N] against model 6 [S->ND,H->N]
D = 2.4965
p = 0.1141
LRT of model 14 [S->D,H->N,D<->N] against model 8 [S->D,H->N]
D = 4.3483
p = 0.037047
LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 11 [S->ND,S->D,H->N]
D = 2.4967
p = 0.11408


In [11]:
"""
The features H->N and S->D (against models 12, 13) find good support when all
three other features are present in the respective null models. This is not
true of S->ND (model 14) and D<->N (model 11).
"""
run_lrt(15, 11, df=1) 
run_lrt(15, 12, df=2)
run_lrt(15, 13, df=2)
run_lrt(15, 14, df=2)

LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 11 [S->ND,S->D,H->N]
D = 2.4967
p = 0.11408
LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 12 [S->ND,S->D,D<->N]
D = 47.126
p = 5.8431e-11
LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 13 [S->ND,H->N,D<->N]
D = 14.066
p = 0.00088235
LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 14 [S->D,H->N,D<->N]
D = 3.0434
p = 0.21834


In [12]:
"""
"""
run_lrt(15, 10, df=4)
run_lrt(15, 5, df=3)

LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 10 [H->N,D<->N]
D = 17.411
p = 0.001608
LRT of model 15 [S->ND,S->D,H->N,D<->N] against model 5 [S->ND,S->D]
D = 95.214
p = 1.6612e-20
